# 02 - Train first-iteration models and predict weak positives

This notebook trains the first-iteration classifiers, evaluates the selected ensemble strategy on the test set, and predicts weak-positive OXPHOS candidates.

The only ensemble retained here is: majority vote within each negative-set group (`negX`) followed by unanimity across negative-set groups.


## 1. Imports and configuration

All paths are relative to the repository root.


In [ ]:
from collections import defaultdict
from pathlib import Path
import pickle
import re

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from sklearn.model_selection import TunedThresholdClassifierCV
except ImportError:
    TunedThresholdClassifierCV = None


RANDOM_SEED = 42
N_JOBS = 4
TRAIN_MODELS = True
WEAK_POSITIVE_PROBABILITY_THRESHOLD = 0.9

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PREPARED_LISTS_DIR = PROJECT_ROOT / "results" / "prepared_lists"
MODEL_DIR = PROJECT_ROOT / "results" / "models"
OUTPUT_DIR = PROJECT_ROOT / "results" / "first_iteration"

EXPRESSION_FILE = DATA_DIR / "boeck_tmm.csv"
STRONG_POSITIVES_FIRST_ITERATION_FILE = DATA_DIR / "strong_positives_first_iteration.csv"
WEAK_POSITIVES_FILE = DATA_DIR / "weak_positives.csv"
TRAINING_SETS_FILE = PREPARED_LISTS_DIR / "01.2_random_training_sets.pkl"
TEST_FILE = PREPARED_LISTS_DIR / "01.1_test.pkl"
TRAINED_MODELS_FILE = MODEL_DIR / "02.1_trained_models_first_iteration.pkl"
UNANIMITY_PREDICTIONS_FILE = OUTPUT_DIR / "02.2_genes_pos_unanimity_with_proba.csv"
STRONG_POSITIVES_SECOND_ITERATION_FILE = DATA_DIR / "02.3_strong_positives_second_iteration.csv"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Helper functions


In [ ]:
def load_expression_matrix(path):
    """Load the expression matrix using WormBase gene IDs as row names."""
    expression = pd.read_csv(path, index_col=1)
    expression = expression.iloc[:, 1:]

    if expression.index.has_duplicates:
        duplicated = expression.index[expression.index.duplicated()].unique()
        raise ValueError(f"Expression matrix contains {len(duplicated)} duplicated gene IDs.")

    return expression


def load_pickle(path):
    with path.open("rb") as file:
        return pickle.load(file)


def save_pickle(obj, path):
    with path.open("wb") as file:
        pickle.dump(obj, file)


def get_model_definitions(random_state=RANDOM_SEED):
    return {
        "RF": RandomForestClassifier(random_state=random_state),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("rbf_svm", SVC(probability=True, random_state=random_state)),
        ]),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier()),
        ]),
    }


def get_param_grids():
    return {
        "RF": {
            "n_estimators": np.arange(1, 20, 1),
            "max_depth": np.arange(1, 20, 1),
            "min_samples_split": np.arange(2, 20, 1),
            "min_samples_leaf": np.arange(1, 20, 1),
        },
        "SVM": {
            "rbf_svm__kernel": ["linear", "poly", "rbf", "sigmoid"],
            "rbf_svm__C": np.arange(0.1, 20, 1),
            "rbf_svm__gamma": np.arange(0.01, 10, 0.5),
        },
        "KNN": {
            "knn__n_neighbors": np.arange(2, 30, 2),
            "knn__weights": ["uniform", "distance"],
            "knn__algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
            "knn__leaf_size": np.arange(1, 100, 10),
        },
    }


In [ ]:
def choose_threshold(best_model, X, y, cv_thr):
    """Choose a decision threshold using CV when available, otherwise a PR curve."""
    if TunedThresholdClassifierCV is not None:
        try:
            tuned = TunedThresholdClassifierCV(
                estimator=best_model,
                scoring="f1",
                cv=cv_thr,
                response_method="predict_proba",
            )
            tuned.fit(X, y)
            return float(tuned.best_threshold_), "StratifiedKFold(10) tuned"
        except ValueError:
            pass

    probabilities = best_model.predict_proba(X)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y, probabilities)
    f1_values = 2 * precision * recall / (precision + recall + 1e-12)
    best_index = np.nanargmax(f1_values)
    threshold = float(thresholds[best_index]) if best_index < len(thresholds) else 0.5
    return threshold, "training precision-recall fallback"


def train_all_models(data_dict, param_grids, subset_keys=None, n_jobs=N_JOBS, random_state=RANDOM_SEED):
    """Train RF, SVM and KNN models for each prepared training collection."""
    trained = {}
    model_definitions = get_model_definitions(random_state=random_state)
    keys_to_train = subset_keys if subset_keys is not None else data_dict.keys()
    cv_hp = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    cv_thr = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_state)

    for model_name, base_model in model_definitions.items():
        print(f"Training model type: {model_name}")
        param_grid = param_grids[model_name]

        for dataset_name in keys_to_train:
            X = data_dict[dataset_name]["train"]["data"]
            y = np.asarray(data_dict[dataset_name]["train"]["classes"])

            grid_search = GridSearchCV(
                base_model,
                param_grid,
                n_jobs=n_jobs,
                cv=cv_hp,
                scoring="f1",
                refit=True,
            )
            grid_search.fit(X, y)
            best_model = grid_search.best_estimator_
            best_cv_f1 = grid_search.best_score_

            threshold, threshold_method = choose_threshold(best_model, X, y, cv_thr)
            probabilities_oof = cross_val_predict(
                best_model,
                X,
                y,
                cv=cv_thr,
                method="predict_proba",
                n_jobs=n_jobs,
            )[:, 1]
            predictions_oof = (probabilities_oof >= threshold).astype(int)

            key = f"{dataset_name}_{model_name}"
            trained[key] = {
                "model": best_model,
                "best_params": grid_search.best_params_,
                "metrics": {
                    "f1": round(f1_score(y, predictions_oof), 3),
                    "accuracy": round(accuracy_score(y, predictions_oof), 3),
                    "roc_auc": round(roc_auc_score(y, probabilities_oof), 3),
                    "threshold": threshold,
                    "cv_f1_params": round(best_cv_f1, 3),
                    "threshold_method": threshold_method,
                },
            }

            print(
                f"{key}: F1={trained[key]['metrics']['f1']:.3f}, "
                f"AUC={trained[key]['metrics']['roc_auc']:.3f}, "
                f"threshold={threshold:.3f}"
            )

    return trained


In [ ]:
def group_model_keys_by_negative_set(trained_models):
    """Group trained model keys by their negX identifier."""
    groups = defaultdict(list)
    pattern = re.compile(r"_neg(\d+)_")

    for model_key in trained_models:
        match = pattern.search(model_key)
        if match:
            groups[match.group(1)].append(model_key)

    if not groups:
        raise ValueError("No negX groups were found in trained model keys.")

    return {neg_id: sorted(keys) for neg_id, keys in sorted(groups.items(), key=lambda item: int(item[0]))}


def predict_majority_within_negx_unanimity_across_negx(trained_models, X):
    """Predict with majority vote within each negX and unanimity across negX groups."""
    groups = group_model_keys_by_negative_set(trained_models)
    group_votes = []
    group_positive_fractions = []
    all_model_probabilities = []

    for neg_id, model_keys in groups.items():
        model_predictions = []

        for model_key in model_keys:
            info = trained_models[model_key]
            probabilities = info["model"].predict_proba(X)[:, 1]
            threshold = float(info["metrics"]["threshold"])
            model_predictions.append((probabilities >= threshold).astype(int))
            all_model_probabilities.append(probabilities)

        model_predictions = np.vstack(model_predictions)
        positive_votes = model_predictions.sum(axis=0)
        majority_threshold = model_predictions.shape[0] // 2 + 1
        group_votes.append((positive_votes >= majority_threshold).astype(int))
        group_positive_fractions.append(positive_votes / model_predictions.shape[0])

    group_votes = np.vstack(group_votes)
    group_positive_fractions = np.vstack(group_positive_fractions)
    all_model_probabilities = np.vstack(all_model_probabilities)

    ensemble_predictions = np.all(group_votes == 1, axis=0).astype(int)
    ensemble_scores = group_positive_fractions.min(axis=0)
    mean_probabilities = all_model_probabilities.mean(axis=0)

    return ensemble_predictions, ensemble_scores, mean_probabilities, group_votes


def evaluate_predictions(y_true, predictions, scores):
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "f1": f1_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, scores),
    }


## 3. Load prepared data


In [ ]:
for path in [EXPRESSION_FILE, STRONG_POSITIVES_FIRST_ITERATION_FILE, WEAK_POSITIVES_FILE, TRAINING_SETS_FILE, TEST_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

expression = load_expression_matrix(EXPRESSION_FILE)
strong_positives_first_iteration = pd.read_csv(STRONG_POSITIVES_FIRST_ITERATION_FILE)
weak_positives = pd.read_csv(WEAK_POSITIVES_FILE)
random_training_sets = load_pickle(TRAINING_SETS_FILE)
test = load_pickle(TEST_FILE)

print(f"Expression matrix: {expression.shape[0]:,} genes x {expression.shape[1]:,} features")
print(f"First-iteration strong positives: {len(strong_positives_first_iteration):,}")
print(f"Weak-positive candidates: {len(weak_positives):,}")
print(f"Prepared training collections: {len(random_training_sets):,}")


## 4. Train or load first-iteration models


In [ ]:
if TRAIN_MODELS:
    trained_models = train_all_models(
        data_dict=random_training_sets,
        param_grids=get_param_grids(),
        n_jobs=N_JOBS,
        random_state=RANDOM_SEED,
    )
    save_pickle(trained_models, TRAINED_MODELS_FILE)
    print(f"Saved trained models to: {TRAINED_MODELS_FILE}")
else:
    trained_models = load_pickle(TRAINED_MODELS_FILE)
    print(f"Loaded trained models from: {TRAINED_MODELS_FILE}")


In [ ]:
trained_models = load_pickle(TRAINED_MODELS_FILE)

## 5. Evaluate the selected ensemble on the held-out test set


In [ ]:
X_test = test["data"]
y_test = np.asarray(test["classes"])
test_genes = test["genes"]

ensemble_predictions, ensemble_scores, mean_probabilities, group_votes = predict_majority_within_negx_unanimity_across_negx(
    trained_models,
    X_test,
)

metrics = evaluate_predictions(y_test, ensemble_predictions, ensemble_scores)
for metric, value in metrics.items():
    print(f"{metric}: {value:.3f}")


## 6. Predict weak-positive candidates


In [ ]:
required_columns = set(strong_positives_first_iteration.columns)
missing_columns = required_columns.difference(weak_positives.columns)
if missing_columns:
    raise ValueError("Missing columns in weak_positives.csv: " + ", ".join(sorted(missing_columns)))

strong_gene_ids = set(strong_positives_first_iteration["gene_ID_worm"].astype(str))
weak_candidates = weak_positives[
    ~weak_positives["gene_ID_worm"].astype(str).isin(strong_gene_ids)
].copy()
weak_candidates = weak_candidates.drop_duplicates(subset="gene_ID_worm", keep="first")

available_weak_candidates = weak_candidates[
    weak_candidates["gene_ID_worm"].astype(str).isin(expression.index.astype(str))
].copy()
weak_candidate_gene_ids = set(available_weak_candidates["gene_ID_worm"].astype(str))
missing_weak_candidates = sorted(
    set(weak_candidates["gene_ID_worm"].astype(str)).difference(expression.index.astype(str))
)

if missing_weak_candidates:
    print(f"Weak-positive candidates absent from the expression matrix: {len(missing_weak_candidates)}")

X_all = expression.to_numpy()
all_gene_ids = expression.index.astype(str).to_numpy()

all_predictions, all_scores, all_mean_probabilities, all_group_votes = predict_majority_within_negx_unanimity_across_negx(
    trained_models,
    X_all,
)

is_weak_positive_candidate = np.array([
    gene in weak_candidate_gene_ids for gene in all_gene_ids
])
passes_probability_threshold = all_mean_probabilities >= WEAK_POSITIVE_PROBABILITY_THRESHOLD
selected_as_weak_positive = (
    is_weak_positive_candidate
    & (all_predictions == 1)
    & passes_probability_threshold
)

unanimity_predictions = pd.DataFrame({
    "Gene": all_gene_ids,
    "Avg_Probability": all_mean_probabilities,
    "Unanimity_Prediction": all_predictions,
    "Passes_Probability_Threshold": passes_probability_threshold,
    "Is_Weak_Positive_Candidate": is_weak_positive_candidate,
    "Selected_As_Weak_Positive": selected_as_weak_positive,
}).sort_values("Avg_Probability", ascending=False)

predicted_weak_gene_ids = unanimity_predictions.loc[
    unanimity_predictions["Selected_As_Weak_Positive"],
    "Gene",
].astype(str).to_numpy()

unanimity_predictions.to_csv(UNANIMITY_PREDICTIONS_FILE, index=False)
print(f"Saved predictions for all expression genes: {len(unanimity_predictions):,}")
print(
    "Weak-positive candidates selected by unanimity "
    f"with Avg_Probability >= {WEAK_POSITIVE_PROBABILITY_THRESHOLD}: "
    f"{len(predicted_weak_gene_ids):,} / {len(available_weak_candidates):,}"
)
print(f"Saved predictions to: {UNANIMITY_PREDICTIONS_FILE}")


## 7. Build the second-iteration strong-positive list


In [ ]:
predicted_weak_rows = available_weak_candidates[
    available_weak_candidates["gene_ID_worm"].astype(str).isin(predicted_weak_gene_ids)
].copy()
predicted_weak_rows = predicted_weak_rows[list(strong_positives_first_iteration.columns)]

strong_positives_second_iteration = pd.concat(
    [strong_positives_first_iteration, predicted_weak_rows],
    ignore_index=True,
)
strong_positives_second_iteration = strong_positives_second_iteration.drop_duplicates(
    subset="gene_ID_worm",
    keep="first",
)

strong_positives_second_iteration.to_csv(STRONG_POSITIVES_SECOND_ITERATION_FILE, index=False)
print(f"Second-iteration strong positives: {len(strong_positives_second_iteration):,}")
print(f"Saved list to: {STRONG_POSITIVES_SECOND_ITERATION_FILE}")
